In [3]:
!pip install datasets
!pip install loralib
!pip install trl
!pip install accelerate
!pip install transformers

In [ ]:
# git clone https://github.com/airobotlab/KoChatGPT
# cp -r  ~/Projects/KoChatGPT/colossalai_ChatGPT_230319/chatgpt ~/Projects/content/chatgpt

In [1]:
import os
import sys

# 1. 홈 디렉토리 경로 자동 변환 + 중간에 겹친 chatgpt/chatgpt 구조 반영
HOME = os.path.expanduser("~")
BASE_PATH = os.path.join(HOME, "Projects/content")
GPT_PATH =  f"{BASE_PATH}/chatgpt"

if BASE_PATH not in sys.path:
    sys.path.append(BASE_PATH)

In [ ]:
modifications = [
    {
        "file": f"{GPT_PATH}/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": f"{GPT_PATH}/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": f"{GPT_PATH}/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": f"{GPT_PATH}/trainer/base.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": f"{GPT_PATH}/trainer/rm.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])


✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/trainer/callbacks/save_checkpoint.py
✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/trainer/strategies/__init__.py
✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/dataset/reward_dataset.py
✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/trainer/base.py
✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/trainer/rm.py


In [16]:
import torch
import transformers
# AutoTokenizer가 한국어 한글의 유니코드(UTF-8) 바이트를 제대로 조합하지 못하고 문자열을 산산조각
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import PreTrainedTokenizerFast
import pandas as pd
import numpy

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))



# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

Torch version:2.10.0
Cuda version: None
transformers version: 5.1.0
GPU 사용 가능여부: False


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

# SKT의 한국어 생성 모델(KoGPT2) pretraied와  토큰나이저
model_name = "skt/kogpt2-base-v2"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2",
  bos_token="</s>", eos_token="</s>", unk_token="<unk>",
  pad_token="<pad>", mask_token="<mask>")
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# 토크나이저 한 번에 자를 수 있는 문장의 최대 길이(토큰 개수)
tokenizer.model_max_length

1000000000000000019884624838656

In [19]:
#  모델은 1번부터 1024번까지만 위치 번호표를 만들 수 있게 짓자
model.config.n_positions

1024

In [20]:
tokenizer.model_max_length = model.config.n_positions # 토크나이저 최대 길이를 모델 뇌 용량 1024로 고정!

In [21]:
# 우리가 자연스럽게 읽는 한국어 문장을 "인공지능 모델이 읽을 수 있는 숫자 배열(ID)"로 어떻게 변환시키는지를 한눈에 확인하는 과정입니다!

input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."
tokens = tokenizer(input_txt).tokens()
print(tokens)
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()

['▁바람', '도', '▁없는', '▁공중에', '▁수직', '의', '▁파', '문을', '▁내', '이며', '▁고', '요', '히', '▁떨어지는', '▁오동', '잎은', '▁누', '구의', '▁발자', '취', '▁입', '니까', '.']


In [22]:
pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kogpt-2_tokens", "Input_IDs"])
df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
kogpt-2_tokens,▁바람,도,▁없는,▁공중에,▁수직,의,▁파,문을,▁내,이며,▁고,요,히,▁떨어지는,▁오동,잎은,▁누,구의,▁발자,취,▁입,니까,.
Input_IDs,10891,7235,9712,49207,14438,8143,9203,9941,9094,9639,9065,8084,8811,21215,34769,19985,9669,10139,21626,8408,9241,23775,389


In [23]:
# 생성되는 전체 문장(질문 포함)의 길이가 최대 128개 조각(토큰)
max_length=128

input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
# 드디어 AI 모델이 일을 시작합니다. 방금 변환한 입력값을 주고 .generate() 함수로 뒷말을 지어내라고 명령합니다.
# do_sample=False: AI의 상상력을 통제하는 아주 중요한 스위치입니다. 
# False로 꺼두면, AI는 매번 다음 단어를 고를 때 무조건 자기가 배운 것 중 "가장 뻔하고 정답일 확률이 1등으로 가장 높은 단어" 한 개만 기계적으로 고르게 됩니다. (이 방식을 욕심쟁이처럼 가장 확률 높은 것만 집어먹는다고 해서 **'그리디 탐색(Greedy Search)'**이라고 부릅니다.)
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
print(tokenizer.decode(output_greedy[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇다면 그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리


In [ ]:
# "여러 갈래의 미래를 동시에 계산(Beam Search)해서 가장 완벽하고 매끄러운 문장"을 지어내도록 인공지능에게 지시하는 똑똑한 문장 생성 코드입니다.

input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)

# num_beams=10 (빔 탐색 10갈래)

# no_repeat_ngram_size=2 (똑같은 말 반복 금지)
# 언어 모델들이 가장 자주 하는 실수인 "그래 그래 그래 그래", "나는 나는 밥을 밥을" 같은 말더듬이 버그를 원천 차단하는 옵션입니다.
# "2개 단어(2-gram)가 똑같이 연달아 2번 반복되면 무조건 오답 처리해라!"라는 엄격한 규칙입니다.

# do_sample=False 랜덤성은 끄고, 수학적으로 가장 완벽한(확률 높은) 길만 걷도록 합니다.
output_beam = model.generate(input_ids, max_length=max_length, num_beams=10, no_repeat_ngram_size=2,
                             do_sample=False)
print(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇지 않습니다."
"어떻게 된 일입니까?"
그녀는 고개를 갸웃거렸다.
"아니, 그게 무슨 말씀이신지 모르겠습니다만."
"무슨 말씀인지 알 수가 없군요."
아무런 대답도 하지 않은 채 그녀는 고개를 끄덕였다.
"그래, 알았어."
그녀의 눈에서 눈물이 주르륵 흘러내렸다.
그녀가 다시 입을 열었다.
"정말 죄송합니다, 고마워요, 고맙습니다"
"


In [26]:
# do_sample=True (창의력 스위치 ON!)
# 이전에는 이게 False로 꺼져 있어서 무조건 확률이 가장 높은(가장 뻔한) 1등 단어만 골랐습니다.
# 방금 이 스위치를 **True**로 켰기 때문에, 이제 모델은 무조건 1등 단어만 고집하지 않고
#  "가끔은 2등이나 3등 단어도 섞어서 문장을 이어가 볼까? 그래야 좀 더 사람 같고 창의적이니까!"라며 
#  약간의 주사위 굴리기(Sampling)를 시작합니다. 똑같은 코드를 두 번 실행하면 매번 다른 대답이 나오게 됩니다.

# top_k=50 (이상한 헛소리 방지!)
# 상상력을 켜주었더니, 가끔 모델이 확률 꼴등(10만 등)인 완전 엉뚱한 외계어 단어를 주사위로 뽑아버리는 대참사가 일어날 수 있습니다.
# 그래서 "아무리 상상력을 발휘해도, 무조건 상위 50등(Top 50) 안에 드는 말이 되는 단어들 중에서만 주사위를 굴려!"라고 
# 안전망을 쳐주는 옵션입니다.

# temperature=2.0
# 이 숫자는 "얼마나 평범함을 거부할 것인가?"를 조절하는 온도 값입니다. (보통 0.1 ~ 2.0 사이를 씁니다.)
# temperature가 0.1 일 때 (차가움): 아주 조심스럽고 보수적입니다. 거의 1등 단어만 뽑는 차가운 로봇 같습니다.
# temperature가 2.0 일 때 (뜨거움 / 약간 미쳤음): 1등 단어와 50등 단어를 뽑을 확률을 거의 비등비등하게 평준화시켜버립니다.
# 아주 과감하게 모험을 떠나서 정말 예측할 수 없는 특이한 문장이나 아주 어색한 헛소리가 나올 확률이 몹시 높아집니다!


output_beam = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, temperature=2.0, top_k=50)
print(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까. 저것을 보면 어디선가 본 듯한데 저것은 무엇인가?"
"그렇겠지요."
이렇게 말하는 이 여인은 눈을 동그랗게 뜨고 그 여인에게 물었다.
"아무래도 저것이 아닌가 하는 생각이 들었습니다."
여인은 이 말을 듣고 가슴이 철렁 내려앉았다.
"무엇 때문에 저런 말을 한 거야? 어째서 저럴까. 왜, 무슨 생각을 하고 있는지 궁금하군."
"왜, 왜 그랬는지, 왜 그렇게 화를 내는지 알 수가 없구


In [28]:
# 누클리어스 샘플링(Nucleus Sampling)"**이라고 불리는 가장 현대적이고 똑똑한 단어 선택 방식(top_p)을 사용해 문장을 지어내는 코드입니다.
# 1. 이전 방식(top_k=50)의 치명적인 단점
# 어제 배웠던 top_k=50은 "무조건 상위 50등까지만 보고 골라라!"라는 무식한 방법이었습니다.

# 상황 A: 정답이 아주 명확할 때 ("대한민국의 수도는 __"). 1등(서울)이 정답일 확률이 99%인데 굳이 50등까지 이상한 단어를 살펴볼 필요가 없습니다. 하지만 top_k는 억지로 50등까지의 쓰레기 단어들을 후보에 올려버려서 가끔 헛소리가 나옵니다.
# 상황 B: 정답이 불확실할 때 ("어제 점심으로 __"). 피자, 햄버거, 짜장면 등 다양한 단어가 올 수 있는데 50등까지만 자르면 너무 뻔한 단어만 나와서 상상력이 부족해집니다.
# 2. 구세주 등장: top_p=0.90 (유동적인 컷오프)
# 이 옵션은 등수(k)로 자르는 게 아니라 **"쓸만한 단어들의 확률을 위에서부터 더해서 총합이 90%(0.90)가 될 때까지만 후보를 추려라!"**라는 아주 똑똑한 규칙입니다.

# 정답이 뻔할 때 (예: 서울 90%, 부산 2% ...): 1등 단어(서울) 딱 하나만 담아도 벌써 90%가 차버리니까 후보 상표를 1개로 확 닫아버립니다. (헛소리 완벽 차단!)
# 정답이 여러 개일 때 (예: 피자 10%, 햄버거 10% ...): 90%를 채우려면 상위 20~30개의 단어를 잔뜩 담아야 합니다. (알아서 후보를 넓혀 상상력을 발휘!)

output_beam = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, top_p=0.90)
print(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."
"그렇지. 그건 그렇고 그걸로 족하오."
"무슨 일입니까?"
나는 고개를 갸우뚱거렸다.
"아니오. 이게 무슨 소린가요? 그게 뭔지 모르겠어요."
그녀는 고개를 끄덕였다.
"그러니까 저는 저를 사랑해 주셨던 거요. 그분도 사랑하셨잖아요. 그래서 저도 저의 사랑을 받아주신 거예요. 그러니까 그분을 사랑한 거죠."
나는


In [ ]:
# "대화형 AI(ChatGPT)를 만들기 위해 사람들이 미리 준비해 둔 '정답 모음집(학습 데이터)' 
# 파일을 파이썬으로 불러와서 읽어 들이는 과정"**입니다.
# 보통 3단계(SFT 👉 RM 👉 PPO)
# SFT 단계

import json
data_path_1_SFT = '/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '불고기용 고기 한우에요?',
  'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  'tokens': 193},
 {'prompt': '쓰던 앱이 유료로 전환됐어',
  'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
  'tokens': 288},
 {'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153}]

In [30]:
data_path_2_RM = '/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl'
with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

10220


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
  'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
  'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
  'completion_2': '라이언에게 말했다.',
  'ranking': [2, 1, 0]},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
  'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
  'completion_1': '이날 목송에서 구글상위노',
  'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
  'ranking': [2, 0, 1]},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
  'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
  'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문

In [ ]:
# "강화학습(PPO)"을 위해 쓸 특별한 시험지 파일을 꺼내오는 과정입가

data_path_3_PPO = '/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl'
with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'}]